# Unified Targeted Feature Explainer

Targeted version of the Delphi autointerp pipeline supporting **Gemma**, **Llama**, and **Qwen** models.
Select a model in the configuration cell below, and the notebook will load the correct
model, features CSV, sparse coders, and layer targets automatically.

All shared logic lives in `explainer_utils/` — no functions are redefined inline.

In [ ]:
# Separate conda environment for delphi (see requirements_explainer.txt).
# Uncomment the lines below if dependencies are not yet installed.
#
# !pip install -e . --ignore-installed blinker
# !pip install plotly
# !pip install --upgrade jupyter ipywidgets
# !pip install zstandard orjson

## Model Selection

Set `SELECTED_MODEL` to one of `"gemma"`, `"llama"`, or `"qwen"`.

In [ ]:
SELECTED_MODEL = "gemma"  # <-- change to "llama" or "qwen" as needed

MODEL_CONFIGS = {
    "gemma": {
        "model": "google/gemma-2-2b",
        "sparse_model": "/scratch/jk87/sg9944/sparsify_mntss_transcoder_gemma22b/",
        "features_path": "/g/data/sz65/sg9944/act_study/lora-finetuning-saes/gemma-2-2b/gemma_features.csv",
        "layer_filter": [8, 9, 10, 11, 12, 13, 14],
        "run_name": "gemma_targeted",
        "output_csv": "gemma_explanations_v1.csv",
    },
    "llama": {
        "model": "meta-llama/Llama-3.2-1B",
        "sparse_model": "/scratch/jk87/sg9944/sparsify_mntss_transcoder_llama321b/",
        "features_path": "results/llama_test/feature_id.csv",
        "layer_filter": [2, 3, 4, 5, 6, 7, 8],
        "run_name": "llama_targeted",
        "output_csv": "llama_explanations_v1.csv",
    },
    "qwen": {
        "model": "Qwen/Qwen3-4B",
        "sparse_model": "/scratch/jk87/sg9944/sparsify_mwhanna_transcoder_qwen34b/",
        "features_path": "./qwen_features.csv",
        "layer_filter": [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
        "run_name": "qwen_targeted",
        "output_csv": "qwen_explanations_v1.csv",
    },
}

cfg = MODEL_CONFIGS[SELECTED_MODEL]
print(f"Selected model: {SELECTED_MODEL} ({cfg['model']})")

In [ ]:
import os
import sys
from pathlib import Path

_repo = Path.cwd()
if (_repo / "explainer_utils").exists():
    sys.path.insert(0, str(_repo))

os.environ['HF_HOME'] = '/g/data4/sz65/sg9944/hugging_face_cache/'
os.environ['HF_HUB_CACHE'] = '/g/data4/sz65/sg9944/hugging_face_cache/hub'
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['HF_DATASETS_CACHE'] = '/scratch/jk87/sg9944/hugging_face_cache_data'
os.environ['HF_OFFLINE'] = '1'
os.environ['TORCHDYNAMO_DISABLE'] = '1'

new_directory_path = '/g/data4/sz65/sg9944/act_study/lora-finetuning-saes/'
os.chdir(new_directory_path)

In [ ]:
import asyncio
import logging
import os
from functools import partial
from pathlib import Path
from typing import Callable
import numpy as np

import json
import orjson

import torch
from torch import Tensor
from transformers import (
    AutoModel,
    AutoTokenizer,
    PreTrainedModel,
    PreTrainedTokenizer,
    PreTrainedTokenizerFast,
)

from delphi import logger
from delphi.clients import Offline, OpenRouter
from delphi.config import CacheConfig, ConstructorConfig, RunConfig, SamplerConfig
from delphi.explainers import ContrastiveExplainer, DefaultExplainer, NoOpExplainer
from delphi.explainers.explainer import ExplainerResult
from delphi.latents import LatentCache, LatentDataset
from delphi.log.result_analysis import get_agg_metrics, load_data, log_results
from delphi.pipeline import Pipe, Pipeline, process_wrapper
from delphi.scorers import DetectionScorer, FuzzingScorer, OpenAISimulator
from delphi.sparse_coders import load_hooks_sparse_coders
from delphi.utils import assert_type, load_tokenized_data

## Target Configuration

Loads hookpoints and feature IDs from the model-specific features CSV.

In [ ]:
from explainer_utils import get_target_layers

FEATURES_PATH = cfg["features_path"]
LAYER_FILTER = cfg["layer_filter"]
TARGET_HOOKPOINTS, TARGET_FEATURES = get_target_layers(FEATURES_PATH, layer_filter=LAYER_FILTER)
DEFAULT_MAX_LATENTS = 100

print(f"Features CSV: {FEATURES_PATH}")
print(f"Layer filter: {LAYER_FILTER}")
print(f"Hookpoints: {TARGET_HOOKPOINTS}")
print(f"Targeted features: { {k: len(v) for k, v in TARGET_FEATURES.items()} }")

## Pipeline Configuration

In [ ]:
cache_cfg = CacheConfig(
    dataset_repo="monology/pile-uncopyrighted",
    dataset_split="train",
    dataset_column="text",
    batch_size=100,
    cache_ctx_len=256,
    n_splits=20,
    n_tokens=200_000,
)

sampler_cfg = SamplerConfig(
    train_type="quantiles",
    test_type="quantiles",
    n_examples_train=40,
    n_examples_test=50,
    n_quantiles=10,
)

constructor_cfg = ConstructorConfig(
    min_examples=50,
    example_ctx_len=32,
    n_non_activating=50,
    non_activating_source="random",
    faiss_embedding_cache_enabled=True,
    faiss_embedding_cache_dir=".embedding_cache",
)

run_cfg = RunConfig(
    name=cfg["run_name"],
    overwrite=["cache", "scores"],
    model=cfg["model"],
    hf_token="YOUR_HF_TOKEN_HERE",
    sparse_model=cfg["sparse_model"],
    hookpoints=TARGET_HOOKPOINTS,
    explainer_model="hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-INT4",
    explainer_model_max_len=4208,
    max_latents=DEFAULT_MAX_LATENTS,
    seed=22,
    num_gpus=torch.cuda.device_count(),
    filter_bos=True,
    verbose=False,
    sampler_cfg=sampler_cfg,
    constructor_cfg=constructor_cfg,
    cache_cfg=cache_cfg,
)

print(f"Model: {run_cfg.model}")
print(f"Run name: {run_cfg.name}")
print(f"Hookpoints: {run_cfg.hookpoints}")
print(f"Targeted features: { {k: len(v) for k, v in TARGET_FEATURES.items()} }")

## Helper Functions

All shared functions are imported from `explainer_utils` — nothing is redefined inline.

In [ ]:
from explainer_utils import (
    DEFAULT_DATASET_CACHE_DIR,
    load_artifacts,
    non_redundant_hookpoints,
    load_tokenized_data_with_cache,
    populate_cache,
    build_latent_dict,
    process_cache,
    get_custom_text_activations,
    print_activation_summary,
    plot_activation_heatmap,
    activations_to_dataframe,
)
DATASET_CACHE_DIR = DEFAULT_DATASET_CACHE_DIR

## Step 1: Load Model & Set Up Paths

In [ ]:
base_path = Path.cwd() / "results"
if run_cfg.name:
    base_path = base_path / run_cfg.name

base_path.mkdir(parents=True, exist_ok=True)
run_cfg.save_json(base_path / "run_config.json", indent=4)

latents_path = base_path / "latents"
explanations_path = base_path / "explanations"
scores_path = base_path / "scores"
neighbours_path = base_path / "neighbours"
visualize_path = base_path / "visualize"

hookpoints, hookpoint_to_sparse_encode, model, transcode = load_artifacts(run_cfg)
tokenizer = AutoTokenizer.from_pretrained(run_cfg.model, token=run_cfg.hf_token)

print(f"Resolved hookpoints: {hookpoints}")

## Step 2: Cache Activations

Caches sparse latent activations to disk. Only needs to run once per model/dataset combination.
Set `"cache"` in `run_cfg.overwrite` to force re-caching.

In [ ]:
nrh = assert_type(
    dict,
    non_redundant_hookpoints(
        hookpoint_to_sparse_encode, latents_path, "cache" in run_cfg.overwrite
    ),
)
if nrh:
    populate_cache(
        run_cfg,
        model,
        nrh,
        latents_path,
        tokenizer,
        transcode,
    )

del model, hookpoint_to_sparse_encode

if run_cfg.constructor_cfg.non_activating_source == "neighbours":
    nrh = assert_type(
        list,
        non_redundant_hookpoints(
            hookpoints, neighbours_path, "neighbours" in run_cfg.overwrite
        ),
    )
    if nrh:
        from delphi.latents.neighbours import NeighbourCalculator
        create_neighbours(run_cfg, latents_path, neighbours_path, nrh)
else:
    print("Skipping neighbour creation")

print("Caching complete.")

## Step 3: Explain & Score Targeted Features

Builds a `latent_dict` mapping each hookpoint to only the targeted feature IDs,
then runs the explain + score pipeline.

In [ ]:
latent_dict = build_latent_dict(hookpoints, TARGET_FEATURES, DEFAULT_MAX_LATENTS)

print("Latent dict summary:")
if latent_dict:
    for hook, ids in latent_dict.items():
        print(f"  {hook}: {len(ids)} features")
else:
    print("  (all features)")

In [ ]:
nrh = assert_type(
    list,
    non_redundant_hookpoints(
        hookpoints, scores_path, "scores" in run_cfg.overwrite
    ),
)
if nrh:
    await process_cache(
        run_cfg,
        latents_path,
        explanations_path,
        scores_path,
        nrh,
        tokenizer,
        latent_dict,
    )

if run_cfg.verbose:
    log_results(scores_path, visualize_path, run_cfg.hookpoints, run_cfg.scorers)

print("Done!")

## Step 4: Analyze Results (Optional)

In [ ]:
latent_df, counts = load_data(scores_path, run_cfg.hookpoints)
processed_df = get_agg_metrics(latent_df, counts)

for score_type, df in processed_df.groupby("score_type"):
    accuracy = df["accuracy"].mean()
    print(f"Score type {score_type}: mean accuracy = {accuracy:.4f}")

## Step 5: Test Activations on Custom Text

Run the model on year-event text strings and see which of the targeted features fire,
how strongly, and on which tokens.

In [ ]:
import pandas as pd

year_events_df = pd.read_csv("./reference_data/year_events.csv")

CUSTOM_TEXTS = year_events_df["text"].tolist()
YEARS = year_events_df["year"].astype(str).tolist()

print(f"Loaded {len(CUSTOM_TEXTS)} year-texts: {YEARS}")

### Reload model for custom text inference

The model was freed after caching. Reload it (and the sparse coders) for custom text probing.

In [ ]:
hookpoints_probe, hookpoint_to_sparse_encode_probe, model_probe, transcode_probe = load_artifacts(run_cfg)
tokenizer_probe = AutoTokenizer.from_pretrained(run_cfg.model, token=run_cfg.hf_token)

print(f"Model reloaded. Hookpoints: {hookpoints_probe}")

In [ ]:
results = get_custom_text_activations(
    texts=CUSTOM_TEXTS,
    model=model_probe,
    tokenizer=tokenizer_probe,
    hookpoint_to_sparse_encode=hookpoint_to_sparse_encode_probe,
    target_features=TARGET_FEATURES,
    transcode=transcode_probe,
)

print_activation_summary(results, top_k=10, activation_threshold=0.0)

In [ ]:
print(f"Active hookpoints: {TARGET_HOOKPOINTS}")

In [ ]:
first_hp = TARGET_HOOKPOINTS[0]

for text_idx in range(len(CUSTOM_TEXTS)):
    plot_activation_heatmap(
        results,
        hookpoint=first_hp,
        text_idx=text_idx,
        max_features=30,
        max_tokens=40,
    )

## Step 6: Aggregate & Export

Flatten activations into a DataFrame, group by (hookpoint, feature, year),
normalise, and export.

In [ ]:
df_activations = activations_to_dataframe(results, YEARS)
print(f"{len(df_activations)} non-zero activations across all texts and hookpoints")

df_grouped = (
    df_activations
    .groupby(["hookpoint", "feature_id", "year"], as_index=False)["activation"]
    .sum()
)

row_max = df_grouped.groupby(["hookpoint", "feature_id"])["activation"].transform("max")
df_grouped["activation_norm"] = df_grouped["activation"] / row_max.replace(0, 1)

print(f"\nGrouped into {len(df_grouped)} (layer, feature, year) combinations")
df_grouped.sort_values("activation", ascending=False).head(20)

In [ ]:
import plotly.graph_objects as go

df_grouped["layer_num"] = df_grouped["hookpoint"].str.replace("layers.", "").astype(int)
df_grouped["layer_feature"] = (
    df_grouped["layer_num"].astype(str) + " / " + df_grouped["feature_id"].astype(str)
)

pivot = df_grouped.pivot_table(
    index=["layer_num", "feature_id", "layer_feature"],
    columns="year",
    values="activation_norm",
    aggfunc="sum",
    fill_value=0,
)

pivot = pivot.sort_index(level=["layer_num", "feature_id"], ascending=[True, True])
pivot = pivot.droplevel(["layer_num", "feature_id"])
pivot = pivot[sorted(pivot.columns)]

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=[str(y) for y in pivot.columns],
    y=pivot.index.tolist(),
    colorscale="Viridis",
    colorbar=dict(title="Normalised Activation"),
))
fig.update_layout(
    title=f"Normalised Feature Activations by Year ({SELECTED_MODEL})",
    xaxis_title="Year",
    yaxis_title="Layer / Feature ID",
    height=max(400, 30 * len(pivot.index) + 100),
    width=max(800, 80 * len(pivot.columns) + 100),
    yaxis=dict(dtick=1, autorange="reversed"),
)
fig.show()

In [ ]:
output_csv = cfg["output_csv"]
df_grouped.to_csv(f"./{output_csv}", index=False)
print(f"Saved to {output_csv}")